In [36]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

In [37]:
torch.manual_seed(100)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [39]:
train_data = pd.read_csv("fashion_train.csv")
test_data = pd.read_csv("fashion_test.csv")

In [40]:
train_data.columns

Index(['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779', 'pixel780',
       'pixel781', 'pixel782', 'pixel783', 'pixel784'],
      dtype='str', length=785)

In [41]:
print("test:" ,test_data.shape)
print("train:" ,train_data.shape)

test: (10000, 785)
train: (60000, 785)


In [42]:
X_train = train_data.drop("label", axis=1)
y_train = train_data["label"]

X_test = test_data.drop("label", axis=1)
y_test = test_data["label"]


In [43]:
X_train = X_train/255.0
X_test = X_test/255.0

In [44]:
X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

X_test = X_test.to_numpy()
y_test = y_test.to_numpy()

In [45]:
class CustomDataset(Dataset) :
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.int64)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index] , self.labels[index]

In [46]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)
print(len(train_dataset))
print(len(test_dataset))

60000
10000


In [47]:
class my_NN(nn.Module):
    def __init__(self, input_dim , output_dim , number_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(number_hidden_layers):
            layers.append(nn.Linear(input_dim,neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer

        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [52]:
def objective(trial):

    #next hyperparameter values for next search space 
    num_hidden_layers = trial.suggest_int("num_hidden_layers" ,1, 5 )
    neuron_per_layer =  trial.suggest_int("neuron_per_layer" ,8, 128,step =8 )
    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128,256])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    #model init
    input_dim = 784
    output_dim = 10

    model = my_NN(input_dim , output_dim , num_hidden_layers, neuron_per_layer, dropout_rate)
    model.to(device)

    #optimizer selecter
    loss_function = nn.CrossEntropyLoss()

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)

    #training loop

    for epoch in range(epochs):

        for batch_features , batch_labels in train_loader:

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
        
            out = model(batch_features)

            loss = loss_function(out,batch_labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

    #evaluation 
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_labels in test_loader:

            batch_features = batch_features.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            outputs = model(batch_features)
        
            _, predicted = torch.max(outputs, 1)
        
            total = total + batch_labels.shape[0]
        
            correct = correct + (predicted == batch_labels).sum().item()

        accuracy = (correct/total)

    return accuracy



In [53]:
print("batches:" , len(train_loader))

batches: 235


In [54]:
import optuna 

study = optuna.create_study(direction ='maximize')


[I 2026-09-02 22:37:19,401] A new study created in memory with name: no-name-af974ec3-257d-44ff-af4e-b1d6e33eb8a3


In [55]:
study.optimize(objective, n_trials=10)

[I 2026-09-02 22:38:15,060] Trial 0 finished with value: 0.8799 and parameters: {'num_hidden_layers': 3, 'neuron_per_layer': 96, 'epochs': 40, 'learning_rate': 0.05745886514918468, 'dropout_rate': 0.2, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.00026177977071454835}. Best is trial 0 with value: 0.8799.
[I 2026-09-02 22:45:00,765] Trial 1 finished with value: 0.8505 and parameters: {'num_hidden_layers': 5, 'neuron_per_layer': 56, 'epochs': 50, 'learning_rate': 0.001676029710068692, 'dropout_rate': 0.4, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.0001261679204305391}. Best is trial 0 with value: 0.8799.
[I 2026-09-02 22:47:33,968] Trial 2 finished with value: 0.8522 and parameters: {'num_hidden_layers': 5, 'neuron_per_layer': 32, 'epochs': 30, 'learning_rate': 0.0010675396207820633, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 7.34754300250575e-05}. Best is trial 0 with value: 0.8799.
[I 2026-09-02 22:49:24,8

In [56]:
study.best_params

{'num_hidden_layers': 2,
 'neuron_per_layer': 88,
 'epochs': 50,
 'learning_rate': 0.0005944839384087616,
 'dropout_rate': 0.4,
 'batch_size': 256,
 'optimizer': 'Adam',
 'weight_decay': 3.078606318658078e-05}

In [57]:
study.best_value

0.8945